# Aula 2 — Notebook 1
## Agente único com ferramentas reais

Neste notebook, vamos avaliar como conectar um modelo de linguagem a um conjunto controlado de ferramentas (tools) e observar como o agente toma decisões de investigação sem receber evidências pré-carregadas. Será nosso ponto de partida para esta aula, incrementando o que vimos construindo na Aula 1.

Resultados esperados:
- Demonstrar o ciclo de request → function_call → execução da tool → retorno ao modelo;
- Produzir um artefato JSON descrevendo diagnóstico, evidências usadas e proposta conceitual de correção;
- Gerar um `trace` com as ferramentas invocadas para permitir rastreabilidade.

Premissas e limites importantes:
- O agente NÃO tem acesso automático ao conteúdo dos arquivos: ele deve usar as tools para obter evidências;
- Este notebook demonstra tomada de decisão automatizada, NÃO execução automática de patches ou commits;
- Sempre valide manualmente qualquer alteração proposta pelo agente antes de aplicar em produção.

Como usar: execute as células sequencialmente. Se uma asserção sobre diretórios falhar, crie `outputs/` no nível do projeto (há uma célula de verificação inicial).

Ao final desta sessão iremos discutir vantagens e riscos de um agente único, e como decompor responsabilidades em agentes especializados.

In [29]:
import os
import json
import textwrap
import subprocess
import sys

from datetime import datetime
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from pathlib import Path

In [30]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
REPO_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / "data"
TARGET_PROJECT_DIR = REPO_ROOT / "target_project/mini_orders_pipeline"
OUTPUT_DIR = PROJECT_ROOT / "outputs"


In [31]:
TARGET_PROJECT_DIR

WindowsPath('c:/Users/cstefano/Desktop/Carlos/Alura/Skills and Go/Repositório/alura-skills-and-go-agentic-engineering/target_project/mini_orders_pipeline')

## Estrutura disponível para o agente

O agente pode consultar três categorias principais de recurso. Cada categoria está representada por funções (tools) que expõem acesso controlado aos dados:

1. **Evidências externas** — tickets, logs, runbooks, histórico e políticas:
   - Arquivos em `data/` (por exemplo `tickets/`, `runbooks/`, `logs/`).
   - A ferramenta de busca TF-IDF (`buscar_evidencias`) permite localizar trechos relevantes sem expor todo o corpus ao modelo de uma só vez.

2. **Projeto-alvo** — o diretório `target_project`:
   - Contém o código e testes do pipeline de pedidos (`src/` e `tests/`).
   - As tools permitem listar arquivos (`listar_arquivos_do_target_project`) ou ler um arquivo específico (`ler_arquivo`) para investigação dirigida.

3. **Ferramentas operacionais** — actions seguras para inspeção e verificação:
   - `buscar_evidencias(consulta, k)`: busca TF-IDF e retorna previews;
   - `ler_arquivo(caminho_relativo)`: retorna o conteúdo de um arquivo;
   - `listar_tickets()`: lista tickets disponíveis;
   - `executar_testes_target_project()`: roda `pytest` no `target_project` e retorna saída e código de retorno (útil para diagnosticar falhas).

Nota de segurança: as tools são projetadas para leitura/inspeção. O agente NÃO modifica arquivos. Quaisquer ações de correção devem ser propostas em linguagem natural e validadas por um humano antes da execução.

Dica pedagógica: peça ao agente que sempre enumere as evidências consultadas (caminhos ou trechos) para facilitar validação e discussões em sala.

In [32]:
EXTENSOES_INDEXADAS = {".md", ".txt", ".log", ".py", ".json", ".toml"}


def ler_texto(caminho):
    return Path(caminho).read_text(encoding="utf-8")


def listar_arquivos_recursivamente(raiz, extensoes=EXTENSOES_INDEXADAS):
    arquivos = []
    for caminho in Path(raiz).rglob("*"):
        if caminho.is_file() and caminho.suffix in extensoes:
            arquivos.append(caminho)
    return sorted(arquivos)


def montar_corpus(data_dir=DATA_DIR, target_project_dir=TARGET_PROJECT_DIR):
    arquivos = listar_arquivos_recursivamente(data_dir) + listar_arquivos_recursivamente(target_project_dir)
    documentos = []

    for caminho in arquivos:
        relativo = caminho.relative_to(REPO_ROOT).as_posix()
        documentos.append({
            "path": relativo,
            "content": ler_texto(caminho),
            "source_type": "target_project" if "target_project" in relativo else "evidence",
        })
    return documentos


def criar_indice_tfidf(documentos):
    vectorizer = TfidfVectorizer(strip_accents="unicode", lowercase=True, ngram_range=(1, 2))
    matriz = vectorizer.fit_transform([doc["content"] for doc in documentos])
    return {"vectorizer": vectorizer, "matrix": matriz, "documents": documentos}


def buscar_no_indice(indice, consulta, k=5):
    consulta_vetor = indice["vectorizer"].transform([consulta])
    scores = cosine_similarity(consulta_vetor, indice["matrix"])[0]
    ordenados = sorted(enumerate(scores), key=lambda item: item[1], reverse=True)[:k]
    resultados = []
    for idx, score in ordenados:
        doc = indice["documents"][idx]
        resultados.append({
            "path": doc["path"],
            "score": round(float(score), 4),
            "source_type": doc["source_type"],
            "preview": doc["content"][:1200],
        })
    return resultados


def listar_tickets():
    return [p.relative_to(REPO_ROOT).as_posix() for p in sorted((DATA_DIR / "tickets").glob("*.md"))]


def carregar_arquivo_relativo(caminho_relativo):
    caminho = REPO_ROOT / caminho_relativo
    if not caminho.exists():
        return json.dumps({"erro": f"Arquivo não encontrado: {caminho_relativo}"}, ensure_ascii=False)
    return caminho.read_text(encoding="utf-8")


def listar_arquivos_do_target_project():
    arquivos = listar_arquivos_recursivamente(TARGET_PROJECT_DIR)
    return [p.relative_to(REPO_ROOT).as_posix() for p in arquivos]


def executar_testes_target_project():
    resultado = subprocess.run(
        [sys.executable, "-m", "pytest", "-q"],
        cwd=TARGET_PROJECT_DIR,
        capture_output=True,
        text=True,
        timeout=30,
    )
    return {
        "returncode": resultado.returncode,
        "stdout": resultado.stdout,
        "stderr": resultado.stderr,
    }


def salvar_json_saida(nome_arquivo, conteudo):
    caminho = OUTPUT_DIR / nome_arquivo
    caminho.parent.mkdir(parents=True, exist_ok=True)
    caminho.write_text(json.dumps(conteudo, ensure_ascii=False, indent=2), encoding="utf-8")
    return caminho.relative_to(REPO_ROOT).as_posix()

DOCUMENTOS = montar_corpus()
INDICE = criar_indice_tfidf(DOCUMENTOS)
print(f"Documentos indexados: {len(DOCUMENTOS)}")


Documentos indexados: 24


## Tools disponíveis

Esta seção explica como as ferramentas são apresentadas ao modelo e como o código mapeia chamadas do modelo para funções Python reais.

1) Esquema para o modelo — `schema_ferramentas_openai()`:
   - Aqui construímos uma lista de objetos JSON descrevendo cada tool (nome, descrição e parâmetros JSON Schema).
   - Esse esquema é passado para `cliente.responses.create(..., tools=...)` para habilitar Function Calling no modelo.

2) Implementações Python — funções no notebook:
   - `listar_tickets()` lê arquivos em `data/tickets/` e retorna a lista;
   - `carregar_arquivo_relativo(caminho_relativo)` (expõe a tool `ler_arquivo`) retorna o conteúdo do arquivo solicitado;
   - `buscar_no_indice(...)` é a implementação por trás da tool `buscar_evidencias`;
   - `listar_arquivos_do_target_project()` percorre `target_project/` e retorna caminhos relativos;
   - `executar_testes_target_project()` roda `pytest` dentro de `target_project` e retorna stdout/stderr e o código de retorno.

3) Dispatcher — `executar_tool_por_nome(nome, argumentos)`:
   - Recebe o `name` e os `arguments` da resposta do modelo (um `function_call`) e executa a função Python correspondente;
   - O resultado é convertido em um `function_call_output` e anexado ao histórico para que o modelo o leia no turno seguinte.

Ponto-chave: a interface entre o modelo e o código é o `name` da função e o JSON de parâmetros. O modelo decide *quando* e *qual* tool usar; o notebook garante que essa chamada seja segura, observável e apenas de leitura.

In [33]:
from openai import OpenAI


def carregar_openai_api_key_de_arquivo():
    candidatos = [
        (Path.cwd() / ".env").resolve(),
        (PROJECT_ROOT / ".env").resolve(),
        (PROJECT_ROOT.parent / ".env").resolve(),
    ]
    vistos = set()

    for caminho in candidatos:
        if caminho in vistos:
            continue
        vistos.add(caminho)

        if not caminho.exists():
            continue

        for linha in caminho.read_text(encoding="utf-8").splitlines():
            conteudo = linha.strip()
            if not conteudo or conteudo.startswith("#") or "=" not in conteudo:
                continue

            chave, valor = conteudo.split("=", 1)
            if chave.strip() != "OPENAI_API_KEY":
                continue

            valor = valor.strip()
            if (valor.startswith('"') and valor.endswith('"')) or (valor.startswith("'") and valor.endswith("'")):
                valor = valor[1:-1]

            if valor:
                os.environ["OPENAI_API_KEY"] = valor
                return caminho.as_posix()

    return None


def exigir_openai_api_key():
    if os.getenv("OPENAI_API_KEY"):
        return

    caminho_carregado = carregar_openai_api_key_de_arquivo()
    if os.getenv("OPENAI_API_KEY"):
        print(f"OPENAI_API_KEY carregada de: {caminho_carregado}")
        return

    caminhos_tentados = [
        (Path.cwd() / ".env").resolve().as_posix(),
        (PROJECT_ROOT / ".env").resolve().as_posix(),
        (PROJECT_ROOT.parent / ".env").resolve().as_posix(),
    ]
    raise RuntimeError(
        "OPENAI_API_KEY não encontrada. Defina a variável de ambiente antes de executar as células com agentes reais, "
        "ou crie um arquivo .env com a linha OPENAI_API_KEY=... em um dos caminhos abaixo:\n"
        + "\n".join([f"- {c}" for c in caminhos_tentados])
    )


def parse_json_seguro(texto):
    try:
        return json.loads(texto)
    except Exception:
        inicio = texto.find("{")
        fim = texto.rfind("}")
        if inicio >= 0 and fim > inicio:
            return json.loads(texto[inicio:fim+1])
        raise


def schema_ferramentas_openai():
    return [
        {
            "type": "function",
            "name": "listar_tickets",
            "description": "Lista os tickets disponíveis para investigação.",
            "parameters": {"type": "object", "properties": {}, "additionalProperties": False},
        },
        {
            "type": "function",
            "name": "ler_arquivo",
            "description": "Lê um arquivo do projeto pelo caminho relativo.",
            "parameters": {
                "type": "object",
                "properties": {"caminho_relativo": {"type": "string"}},
                "required": ["caminho_relativo"],
                "additionalProperties": False,
            },
        },
        {
            "type": "function",
            "name": "buscar_evidencias",
            "description": "Busca evidências em tickets, logs, runbooks, histórico, políticas e código do target_project usando TF-IDF.",
            "parameters": {
                "type": "object",
                "properties": {
                    "consulta": {"type": "string"},
                    "k": {"type": "integer", "minimum": 1, "maximum": 8},
                },
                "required": ["consulta"],
                "additionalProperties": False,
            },
        },
        {
            "type": "function",
            "name": "listar_arquivos_do_target_project",
            "description": "Lista arquivos disponíveis no projeto-alvo que pode conter o código afetado pelo incidente.",
            "parameters": {"type": "object", "properties": {}, "additionalProperties": False},
        },
        {
            "type": "function",
            "name": "executar_testes_target_project",
            "description": "Executa a suíte de testes do target_project para observar o estado atual do projeto.",
            "parameters": {"type": "object", "properties": {}, "additionalProperties": False},
        },
    ]


def executar_tool_por_nome(nome, argumentos):
    if nome == "listar_tickets":
        return listar_tickets()
    if nome == "ler_arquivo":
        return carregar_arquivo_relativo(argumentos["caminho_relativo"] )
    if nome == "buscar_evidencias":
        return buscar_no_indice(INDICE, argumentos["consulta"], argumentos.get("k", 5))
    if nome == "listar_arquivos_do_target_project":
        return listar_arquivos_do_target_project()
    if nome == "executar_testes_target_project":
        return executar_testes_target_project()
    return {"erro": f"Tool desconhecida: {nome}"}


def executar_agente_openai(mensagem_usuario, instrucoes_sistema, modelo="gpt-4.1-mini", max_turnos=8):
    exigir_openai_api_key()
    cliente = OpenAI()
    historico = [{"role": "user", "content": mensagem_usuario}]
    trace = []

    for turno in range(max_turnos):
        resposta = cliente.responses.create(
            model=modelo,
            instructions=instrucoes_sistema,
            input=historico,
            tools=schema_ferramentas_openai(),
            tool_choice="auto",
        )

        chamadas = [item for item in resposta.output if item.type == "function_call"]
        textos = [item for item in resposta.output if item.type == "message"]

        if not chamadas:
            texto_final = "\n".join([c.text for msg in textos for c in msg.content if c.type == "output_text"])
            return {"final": texto_final, "trace": trace}

        historico += resposta.output

        for chamada in chamadas:
            argumentos = json.loads(chamada.arguments or "{}")
            resultado = executar_tool_por_nome(chamada.name, argumentos)
            trace.append({"turno": turno + 1, "tool": chamada.name, "arguments": argumentos, "result_preview": str(resultado)[:800]})
            historico.append({
                "type": "function_call_output",
                "call_id": chamada.call_id,
                "output": json.dumps(resultado, ensure_ascii=False),
            })

    raise RuntimeError("O agente excedeu o número máximo de turnos de uso de ferramentas.")

## Contrato de saída

O agente deve produzir uma resposta estritamente em JSON seguindo o formato `CONTRATO_SAIDA` definido na célula seguinte. Esse contrato facilita:

- Validação automática do formato de saída;
- Registro e comparação de investigações entre execuções;
- Extração de evidências e caminhos que o agente consultou (útil para revisão humana).

Boas práticas ao avaliar a saída do agente:
- Verifique se `diagnostico.evidencias` contém caminhos ou trechos que de fato apoiam a hipótese;
- Cheque o campo `confianca` e peça ao agente para relacionar evidências que sustentem essa confiança;
- Confirme que `modulos_afetados` referencia arquivos existentes no `target_project` antes de aceitar mudanças propostas.

In [34]:
CONTRATO_SAIDA = {
    "incident_id": "identificador do incidente investigado",
    "diagnostico": {
        "causa_provavel": "texto curto",
        "confianca": "baixa | media | alta",
        "evidencias": ["lista de caminhos de arquivos ou trechos usados"],
    },
    "modulos_afetados": ["arquivos do target_project possivelmente afetados"],
    "proposta_conceitual": {
        "descricao": "alteração sugerida em linguagem natural",
        "tipo_mudanca": "validacao | transformacao | logging | teste | outro",
        "nao_executar_automaticamente": True,
    },
    "plano_de_testes": ["testes que deveriam ser adicionados ou executados"],
    "decisao": {
        "pode_preparar_patch_na_aula_4": "sim | nao",
        "motivo": "justificativa curta",
        "requer_revisao_humana": True,
    },
}

## Execução do agente único

Nesta execução, solicitamos que o agente investigue um ticket específico (por exemplo `INC-001`). Importante notar:

- O agente começa sem evidências e decide quais ferramentas invocar;
- Cada invocação é registrada no `trace` (turno, tool, argumentos e um preview do resultado);
- O loop de interação termina quando o modelo não retorna mais `function_call`s e emite uma resposta final em texto JSON;

Como interpretar o `trace`:
- Leia o `trace` sequencialmente para entender a cadeia de raciocínio do agente;
- Para cada chamada, abra o arquivo referenciado (usando `ler_arquivo`) e verifique se o trecho citado corresponde ao preview retornado;
- Use o `trace` para construir perguntas de follow-up e validar hipóteses com base nas evidências reais.

In [35]:
TARGET_PROJECT_DIR

WindowsPath('c:/Users/cstefano/Desktop/Carlos/Alura/Skills and Go/Repositório/alura-skills-and-go-agentic-engineering/target_project/mini_orders_pipeline')

In [39]:
instrucoes = f"""
Você é um agente técnico de investigação de incidentes em pipelines de dados.

Você deve investigar o incidente solicitado usando as tools disponíveis.
Não invente evidências. Se precisar consultar tickets, logs, runbooks, políticas, histórico ou código, use as tools.

Seu objetivo é produzir uma proposta conceitual de correção, não um patch executável.
Não altere arquivos do target_project.

Responda apenas com JSON válido seguindo esta estrutura conceitual:
{json.dumps(CONTRATO_SAIDA, ensure_ascii=False, indent=2)}

Para classificar a confiança da causa provável, use:
- baixa: evidências indiretas ou inconclusivas
- media: evidências razoáveis, mas não definitivas
- alta: evidências diretas e consistentes
"""

mensagem = """
Investigue o ticket INC-002.
Identifique a causa provável, os módulos afetados no target_project,
a proposta conceitual de correção e um plano mínimo de testes.
"""

resultado = executar_agente_openai(mensagem, instrucoes)
print(resultado["final"])

{
  "incident_id": "INC-002",
  "diagnostico": {
    "causa_provavel": "order_total chega como string com vírgula decimal, e a transformação tenta converter usando float direto, causando métrica incorreta",
    "confianca": "alta",
    "evidencias": [
      "Aula 2/data/tickets/INC-002-order-total-string.md",
      "target_project/mini_orders_pipeline/src/mini_orders_pipeline/transformacao_pedidos.py",
      "Aula 2/data/runbooks/revenue_transformation.md"
    ]
  },
  "modulos_afetados": [
    "target_project/mini_orders_pipeline/src/mini_orders_pipeline/transformacao_pedidos.py"
  ],
  "proposta_conceitual": {
    "descricao": "Alterar a função de transformação para normalizar o campo order_total convertendo strings com vírgula decimal em número float corretamente, rejeitando valores inválidos com erro estruturado para evitar métricas incorretas.",
    "tipo_mudanca": "transformacao",
    "nao_executar_automaticamente": true
  },
  "plano_de_testes": [
    "Testar transformação com o

## Rastro de uso das ferramentas

O trace abaixo mostra quais tools o agente decidiu usar. Este rastro é importante para discutir rastreabilidade: uma conclusão técnica deve estar conectada às fontes consultadas.

In [40]:
resultado

{'final': '{\n  "incident_id": "INC-002",\n  "diagnostico": {\n    "causa_provavel": "order_total chega como string com vírgula decimal, e a transformação tenta converter usando float direto, causando métrica incorreta",\n    "confianca": "alta",\n    "evidencias": [\n      "Aula 2/data/tickets/INC-002-order-total-string.md",\n      "target_project/mini_orders_pipeline/src/mini_orders_pipeline/transformacao_pedidos.py",\n      "Aula 2/data/runbooks/revenue_transformation.md"\n    ]\n  },\n  "modulos_afetados": [\n    "target_project/mini_orders_pipeline/src/mini_orders_pipeline/transformacao_pedidos.py"\n  ],\n  "proposta_conceitual": {\n    "descricao": "Alterar a função de transformação para normalizar o campo order_total convertendo strings com vírgula decimal em número float corretamente, rejeitando valores inválidos com erro estruturado para evitar métricas incorretas.",\n    "tipo_mudanca": "transformacao",\n    "nao_executar_automaticamente": true\n  },\n  "plano_de_testes": [\n

In [37]:
for passo in resultado["trace"]:
    print(json.dumps(passo, ensure_ascii=False, indent=2))
    print("-" * 80)

{
  "turno": 1,
  "tool": "listar_tickets",
  "arguments": {},
  "result_preview": "['Aula 2/data/tickets/INC-001-schema-customer-id.md', 'Aula 2/data/tickets/INC-002-order-total-string.md', 'Aula 2/data/tickets/INC-003-date-format.md']"
}
--------------------------------------------------------------------------------
{
  "turno": 2,
  "tool": "ler_arquivo",
  "arguments": {
    "caminho_relativo": "Aula 2/data/tickets/INC-001-schema-customer-id.md"
  },
  "result_preview": "# INC-001 — Falha na ingestão de pedidos\n\n**Sistema:** mini_orders_pipeline\n**Severidade inicial:** alta\n**Reportado por:** monitoramento de pipeline\n\n## Sintoma\nO job `orders_daily_ingestion` falhou durante a etapa de ingestão de pedidos. O erro começou após a entrada de um novo lote enviado pelo parceiro `marketplace_beta`.\n\n## Mensagem observada\n`KeyError: 'customer_id'`\n\n## Impacto\nPedidos do parceiro não foram processados na janela esperada. O dashboard diário de receita ficou incompleto.\n\n## P

## Salvando o artefato

Vamos tentar converter a resposta em JSON e salvar a investigação em `outputs/reports/`. Nas próximas aulas, artefatos como esse poderão alimentar avaliação, revisão, patch sugerido e rascunho de PR.

In [38]:
artefato = parse_json_seguro(resultado["final"])
caminho = salvar_json_saida("reports/incidente_inc001_agente_unico.json", artefato)
print("Arquivo salvo em:", caminho)

Arquivo salvo em: Aula 2/outputs/reports/incidente_inc001_agente_unico.json


## Limites observáveis de um agente único

Um agente único pode fazer muita coisa: buscar evidências, ler código, interpretar logs e sugerir ações. O problema é que ele também acumula muitas responsabilidades.

Ao revisar o resultado, observe:

- ele separou claramente evidência de hipótese?
- leu arquivos suficientes do `target_project`?
- sugeriu testes coerentes com a causa provável?
- indicou revisão humana antes de qualquer alteração?
- deixou claro o que ainda não sabe?

Essas perguntas motivam o próximo notebook: decompor a investigação em agentes especializados.